In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 0. Reproducibility
# ============================================================

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


# ============================================================
# 1. Load MovieLens 1M
# ============================================================

data_folder = "datasets/movie_lens/"

ratings_file = os.path.join(data_folder, "ratings.dat")
movies_file = os.path.join(data_folder, "movies.dat")
users_file = os.path.join(data_folder, "users.dat")


ratings_df = pd.read_csv(
    ratings_file,
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies_df = pd.read_csv(
    movies_file,
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

users_df = pd.read_csv(
    users_file,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)


print("Ratings:", len(ratings_df))
print("Users:", ratings_df["user_id"].nunique())
print("Movies:", ratings_df["movie_id"].nunique())

print(movies_df.head())

Ratings: 1000209
Users: 6040
Movies: 3706
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy


In [2]:
# ============================================================
# 2. Map raw IDs -> contiguous indices
# ============================================================

unique_users = ratings_df["user_id"].unique()
unique_movies = ratings_df["movie_id"].unique()


user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(unique_users)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(unique_movies)
}


# Reverse mappings
idx_to_user = {
    idx: user_id
    for user_id, idx in user_to_idx.items()
}

idx_to_movie = {
    idx: movie_id
    for movie_id, idx in movie_to_idx.items()
}


ratings_df["user_idx"] = ratings_df["user_id"].map(user_to_idx)
ratings_df["movie_idx"] = ratings_df["movie_id"].map(movie_to_idx)


num_users = len(user_to_idx)
num_movies = len(movie_to_idx)

print("num_users:", num_users)
print("num_movies:", num_movies)

num_users: 6040
num_movies: 3706


In [3]:
# ============================================================
# 3. Chronological split
# ============================================================

ratings_df = ratings_df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)


ratings_df["rank_from_end"] = (
    ratings_df.groupby("user_id")
    .cumcount(ascending=False)
)


test_df = ratings_df[
    ratings_df["rank_from_end"] == 0
].copy()

val_df = ratings_df[
    ratings_df["rank_from_end"] == 1
].copy()

train_df = ratings_df[
    ratings_df["rank_from_end"] >= 2
].copy()


for df in [train_df, val_df, test_df]:
    df.drop(columns=["rank_from_end"], inplace=True)


print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (988129, 6)
Val: (6040, 6)
Test: (6040, 6)


In [4]:
# ============================================================
# 4. Dataset + DataLoaders
# ============================================================

class MovieLensDataset(Dataset):

    def __init__(self, df):

        self.user_ids = torch.tensor(
            df["user_idx"].values,
            dtype=torch.long
        )

        self.movie_ids = torch.tensor(
            df["movie_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            df["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.user_ids[idx],
            self.movie_ids[idx],
            self.ratings[idx]
        )


train_dataset = MovieLensDataset(train_df)
val_dataset = MovieLensDataset(val_df)
test_dataset = MovieLensDataset(test_df)


train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

In [5]:
# ============================================================
# 5. Simple user/movie embedding + MLP model
# ============================================================

class SimpleMLP(nn.Module):

    def __init__(self, num_users, num_movies, d_embedding):
        super().__init__()

        self.user_emb = nn.Embedding(
            num_users,
            d_embedding
        )

        self.movie_emb = nn.Embedding(
            num_movies,
            d_embedding
        )

        self.mlp = nn.Sequential(
            nn.Linear(d_embedding * 2, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, user_ids, movie_ids):

        # [B, D]
        user_vec = self.user_emb(user_ids)

        # [B, D]
        movie_vec = self.movie_emb(movie_ids)

        # [B, 2D]
        x = torch.cat(
            [user_vec, movie_vec],
            dim=1
        )

        # [B, 1]
        pred = self.mlp(x)

        # [B]
        return pred.squeeze(1)

In [6]:
# ============================================================
# 6. Train recommender
# ============================================================

embedding_dim = 64

model = SimpleMLP(
    num_users=num_users,
    num_movies=num_movies,
    d_embedding=embedding_dim
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for user_batch, movie_batch, rating_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(
            user_batch,
            movie_batch
        )

        loss = criterion(
            predictions,
            rating_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train MSE: {avg_train_loss:.4f}"
    )

Epoch 01 | Train MSE: 0.9944
Epoch 02 | Train MSE: 0.8366
Epoch 03 | Train MSE: 0.8051
Epoch 04 | Train MSE: 0.7755
Epoch 05 | Train MSE: 0.7455
Epoch 06 | Train MSE: 0.7170
Epoch 07 | Train MSE: 0.6901
Epoch 08 | Train MSE: 0.6646
Epoch 09 | Train MSE: 0.6402
Epoch 10 | Train MSE: 0.6165


In [7]:
# ============================================================
# 7. Evaluate recommender
# ============================================================

model.eval()

total_val_loss = 0.0

with torch.no_grad():

    for user_batch, movie_batch, rating_batch in val_loader:

        predictions = model(
            user_batch,
            movie_batch
        )

        loss = criterion(
            predictions,
            rating_batch
        )

        total_val_loss += loss.item()


avg_val_loss = total_val_loss / len(val_loader)

print("Validation MSE:", avg_val_loss)

Validation MSE: 0.922743042310079


In [8]:
# ============================================================
# 8. Extract learned movie embeddings
# ============================================================

movie_vectors = (
    model
    .movie_emb
    .weight
    .detach()
    .clone()
)

print(movie_vectors.shape)

# Expected:
# [num_movies, 64]

torch.Size([3706, 64])


In [21]:
# ============================================================
# 9. Vector Quantizer
# ============================================================

class ResidualVectorQuantizer(nn.Module):

    def __init__(self, codebook_size, d_embedding):
        super().__init__()

        # Shape: [K, D]
        self.n_codebooks = 4 
        self.codebooks = [] 
        
        for n in range(self.n_codebooks): 
            self.codebooks.append(nn.Embedding(
                codebook_size,
                d_embedding
            )) 

        self.codebooks = nn.ModuleList(self.codebooks) 

    def forward(self, x):

        # x:
        # [B, D]

        # ----------------------------------------------------
        # Compare every vector against every codebook vector
        #
        # x[:, None, :]
        # [B, 1, D]
        #
        # codebook.weight[None, :, :]
        # [1, K, D]
        #
        # diff
        # [B, K,  ]





        
        res = x 
        code_ids = [] 
        quantized_x = torch.zeros_like(x)
        # ------------------------------nbh----------------------
        for i, code_level in enumerate(self.codebooks):

            diff = (
                res[:, None, :]
                - self.codebooks[i].weight[None, :, :]
            )
    
            # Squared Euclidean distance
            #
            # [B, K, D]
            #      ↓ sum over D
            # [B, K]
    
            distances = (diff ** 2).sum(dim=2)
    
            # For every movie, select closest code
            #
            # [B, K] -> [B]
                
            code_id_at_level_l = torch.argmin(
                distances,
                dim=1
            )
            code_ids.append(code_id_at_level_l)
    
            # Retrieve selected vectors
            #
            # [B]
    
            quantized = self.codebooks[i](code_id_at_level_l)
            quantized_x = quantized_x + quantized
            
            res = res - quantized 

        # quantized: [B, D]
        # code_ids:  [B]
        code_ids = torch.stack(code_ids, dim=1)

        return quantized_x, code_ids
codebook_size = 5

quantizer = ResidualVectorQuantizer(
    codebook_size=codebook_size,
    d_embedding=embedding_dim
)

quantizer(torch.randn(25, embedding_dim))
    

(tensor([[-0.5501, -1.0767,  1.3505,  ..., -1.1024,  0.2903,  2.3384],
         [ 0.1352,  1.8004, -0.7610,  ..., -0.6171,  0.8715,  0.2792],
         [ 0.3513, -0.6470, -0.7165,  ..., -1.0556, -0.2881,  0.4784],
         ...,
         [ 2.3010, -1.3714,  0.6105,  ..., -0.6306,  2.3170,  2.7896],
         [-0.5501, -1.0767,  1.3505,  ..., -1.1024,  0.2903,  2.3384],
         [ 0.2780, -2.2068,  0.0714,  ...,  0.3042,  0.2172,  1.3701]],
        grad_fn=<AddBackward0>),
 tensor([[0, 4, 0, 2],
         [0, 4, 2, 2],
         [3, 4, 2, 2],
         [0, 4, 4, 2],
         [3, 2, 3, 0],
         [3, 4, 2, 2],
         [3, 1, 4, 2],
         [0, 1, 4, 2],
         [3, 0, 0, 0],
         [0, 4, 2, 2],
         [3, 1, 2, 2],
         [3, 4, 2, 2],
         [4, 4, 2, 2],
         [3, 4, 2, 2],
         [3, 2, 4, 1],
         [4, 0, 0, 0],
         [3, 1, 3, 0],
         [3, 0, 0, 0],
         [4, 3, 0, 3],
         [3, 0, 4, 0],
         [4, 2, 0, 0],
         [3, 1, 4, 2],
         [3, 4, 4, 0

In [23]:
# ============================================================
# 10. Train Residual Vector Quantizer
# ============================================================
# movie embedding x
#       ↓
# RVQ
#       ↓
# [q0, q1, q2, q3]
codebook_size = 5

quantizer = ResidualVectorQuantizer(
    codebook_size=codebook_size,
    d_embedding=embedding_dim
)


# Fixed input embeddings
quantizer_train = movie_vectors.detach()


# Initialize codebook using actual random movie embeddings
# instead of arbitrary random vectors.

# with torch.no_grad():

#     random_idx = torch.randperm(
#         len(quantizer_train)
#     )[:codebook_size]

#     quantizer.codebook.weight.copy_(
#         quantizer_train[random_idx]
#     )


vq_optimizer = torch.optim.Adam(
    quantizer.parameters(),
    lr=0.001
)

vq_criterion = nn.MSELoss()


num_vq_epochs = 100


for epoch in range(num_vq_epochs):

    vq_optimizer.zero_grad()

    quantized, code_ids = quantizer(
        quantizer_train
    )

    # Move codebook vectors toward
    # the movie embeddings assigned to them
    vq_loss = vq_criterion(
        quantized,
        quantizer_train
    )

    vq_loss.backward()
    vq_optimizer.step()


    if epoch % 10 == 0:
    
        print(
            f"Epoch {epoch:03d} | "
            f"loss={vq_loss.item():.4f}"
        )
    
        for level in range(code_ids.shape[1]):
    
            counts = torch.bincount(
                code_ids[:, level],
                minlength=codebook_size
            )
    
            print(
                f"  Level {level}: "
                f"{counts.tolist()}"
            )

Epoch 000 | loss=3.3789
  Level 0: [1078, 744, 903, 618, 363]
  Level 1: [987, 209, 126, 83, 2301]
  Level 2: [202, 47, 919, 1390, 1148]
  Level 3: [259, 1751, 1380, 276, 40]
Epoch 010 | loss=3.3194
  Level 0: [1078, 749, 907, 611, 361]
  Level 1: [970, 213, 128, 79, 2316]
  Level 2: [196, 47, 940, 1382, 1141]
  Level 3: [252, 1759, 1388, 269, 38]
Epoch 020 | loss=3.2622
  Level 0: [1072, 750, 913, 609, 362]
  Level 1: [960, 217, 128, 75, 2326]
  Level 2: [180, 50, 958, 1392, 1126]
  Level 3: [242, 1769, 1403, 255, 37]
Epoch 030 | loss=3.2068
  Level 0: [1078, 745, 921, 606, 356]
  Level 1: [951, 214, 130, 72, 2339]
  Level 2: [176, 52, 965, 1392, 1121]
  Level 3: [229, 1779, 1414, 245, 39]
Epoch 040 | loss=3.1530
  Level 0: [1084, 746, 920, 596, 360]
  Level 1: [940, 215, 130, 70, 2351]
  Level 2: [168, 53, 976, 1397, 1112]
  Level 3: [220, 1805, 1411, 233, 37]
Epoch 050 | loss=3.1016
  Level 0: [1087, 751, 920, 584, 364]
  Level 1: [924, 215, 131, 68, 2368]
  Level 2: [161, 54, 980, 

In [31]:
# ============================================================
# 11. Assign semantic code to each movie
# ============================================================

quantizer.eval()

with torch.no_grad():

    quantized_movie_vectors, movie_code_ids = quantizer(
        movie_vectors
    )


print(movie_code_ids.shape)

for level in range(movie_code_ids.shape[1]):
    counts = torch.bincount(
        movie_code_ids[:, level],
        minlength=codebook_size
    )
    print(f"Level {level}: {counts.tolist()}")


torch.Size([3706, 4])
Level 0: [1108, 750, 916, 565, 367]
Level 1: [904, 233, 135, 53, 2381]
Level 2: [127, 60, 1095, 1415, 1009]
Level 3: [183, 1952, 1376, 162, 33]


In [32]:
# ============================================================
# 12. Inspect movies assigned to each code
# ============================================================
level = 0  # Change to 1, 2, or 3 to inspect other levels

for code in range(codebook_size):
    internal_indices = torch.where(
        movie_code_ids[:, level] == code
    )[0]

    # Convert internal index -> original MovieLens movie ID
    raw_movie_ids = [
        idx_to_movie[idx.item()]
        for idx in internal_indices
    ]


    # Look up metadata
    code_movies = movies_df[
        movies_df["movie_id"].isin(raw_movie_ids)
    ][
        ["movie_id", "title", "genres"]
    ]


    print()
    print("=" * 80)
    print(
        f"CODE {code} | "
        f"{len(code_movies)} movies"
    )
    print("=" * 80)

    print(
        code_movies.head(20).to_string(index=False)
    )


CODE 0 | 1108 movies
 movie_id                                                title                    genres
        3                              Grumpier Old Men (1995)            Comedy|Romance
        9                                  Sudden Death (1995)                    Action
       10                                     GoldenEye (1995) Action|Adventure|Thriller
       11                       American President, The (1995)      Comedy|Drama|Romance
       13                                         Balto (1995)      Animation|Children's
       23                                     Assassins (1995)                  Thriller
       26                                       Othello (1995)                     Drama
       29                    City of Lost Children, The (1995)          Adventure|Sci-Fi
       30 Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)                     Drama
       40                      Cry, the Beloved Country (1995)                     Drama